# 🚗 Car Market Segmentation — Unsupervised Clustering

## 📌 Project Overview

Car manufacturers, dealerships, and analysts often want to understand 
how vehicles naturally group together based on their specs and pricing — 
without predefined categories like "economy" or "luxury." Rather than 
relying on manual labels, we can let the data itself reveal these 
groupings.

In this project, we apply **K-Means clustering**, an unsupervised 
learning technique, to a dataset of ~12,000 U.S. car models 
(1990–2018), using features like price (MSRP), horsepower, fuel 
efficiency, and engine specs to uncover natural market segments.

---

## 🎯 Objective

Group cars into distinct segments based on their specifications and 
price, **without using any predefined labels**, then interpret what 
characterizes each resulting group (e.g., "high-power luxury," 
"budget-efficient compacts," "family SUVs").

---

## 🔑 Key Difference From Supervised Learning

Unlike our previous regression and classification projects, this 
project has **no target variable**. There's nothing to predict and no 
"correct answer" to check against. Instead of a train/test split and 
accuracy scores, we:

- Skip the train/test split entirely — clustering uses the full 
  dataset, since there's no prediction to validate against unseen data
- Judge success through **cluster separation and interpretability**, 
  using tools like the Elbow Method and Silhouette Score, rather than 
  accuracy or F1
- Focus on **describing and understanding** the groups the algorithm 
  finds, rather than predicting a known outcome

---

## 🗺️ Project Roadmap

1. **Data Collection** — car specs and pricing data via the Kaggle API
2. **Data Cleaning** — handle missing values, duplicates, and 
   inconsistent entries
3. **Exploratory Data Analysis** — understand distributions of price, 
   horsepower, MPG, and other specs
4. **Feature Scaling** — critical for K-Means, since it's a 
   distance-based algorithm sensitive to feature magnitude
5. **Finding Optimal K** — Elbow Method and Silhouette Score to decide 
   how many clusters best fit the data
6. **Fitting K-Means** — assign each car to a cluster
7. **Visualizing Clusters** — scatter plots and dimensionality 
   reduction (PCA) to see the groupings
8. **Profiling Clusters** — describe what makes each segment distinct 
   (e.g., average price, power, size, common vehicle styles)

---

## 📊 Data Source

This project uses the **Car Features and MSRP** dataset from Kaggle, 
containing car listings from 1990–2018, including make, model, engine 
specs, MPG, vehicle category, and MSRP.

> Source: [Car Features and MSRP — Kaggle](https://www.kaggle.com/datasets/CooperUnion/cardataset)

#### 📥 Loading the Dataset

We read the located CSV into a pandas DataFrame — this is our raw, 
unprocessed dataset, straight from the Kaggle download.

In [20]:
%pip install pandas numpy matplotlib seaborn scikit-learn jupyterlab nbconvert kagglehub sweetviz joblib yellowbrick setuptools watermark -q

Note: you may need to restart the kernel to use updated packages.


In [21]:
# ==========================================
# 1. DATA MANIPULATION & VISUALIZATION
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sweetviz as sv
import os
import kagglehub
import logging
import warnings
import kagglehub
from sweetviz import FeatureConfig

warnings.filterwarnings('ignore', category=UserWarning)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ==========================================
# 2. PREPROCESSING & PIPELINES
# ==========================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# ==========================================
# 3. CLUSTERING & EVALUATION
# ==========================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer

In [22]:
# Load the watermark extension to log the environment state
%reload_ext watermark

# Display professional metadata tracking our data engineering stack (NO SPACES after commas)
%watermark -a "Maykon - Car Market Segmentation 🚗" -d -u -v -p pandas,numpy,matplotlib,seaborn,scikit-learn,kagglehub,jupyterlab,nbconvert,sweetviz,joblib,yellowbrick,watermark,setuptools

Author: Maykon - Car Market Segmentation 🚗

Last updated: 2026-08-15

Python implementation: CPython
Python version       : 3.13.7
IPython version      : 9.16.1

pandas      : 3.0.5
numpy       : 2.5.2
matplotlib  : 3.11.1
seaborn     : 0.13.2
scikit-learn: 1.9.0
kagglehub   : 1.0.2
jupyterlab  : 4.6.3
nbconvert   : 7.17.1
sweetviz    : 2.3.3
joblib      : 1.5.3
yellowbrick : 1.5
watermark   : 2.6.0
setuptools  : 84.0.0



In [23]:
# --- AUTOMATED KAGGLE INGESTION ---
# Download the latest version of the specific student behavioral dataset
path = kagglehub.dataset_download("CooperUnion/cardataset")
print("🚀 Path to dataset files:", path)

🚀 Path to dataset files: C:\Users\LarTI\.cache\kagglehub\datasets\CooperUnion\cardataset\versions\1


In [24]:
# --- LOCATING AND READING THE CSV ---
# List out all files inside the downloaded repository path to spot the target file
all_files = os.listdir(path)
print("📂 Files discovered in directory:", all_files)

# Filter out all CSV files dynamically
csv_files = [file for file in all_files if file.endswith('.csv')]

if len(csv_files) == 0:
    raise FileNotFoundError("❌ Critical Error: No CSV files found in the downloaded folder!")
else:
    # Grab the primary CSV file found
    csv_filename = csv_files[0]
    full_csv_path = os.path.join(path, csv_filename)
    print(f"🎯 Target CSV located: {csv_filename}")

📂 Files discovered in directory: ['data.csv']
🎯 Target CSV located: data.csv


In [25]:
# Ingest the dataset into a pandas DataFrame
df = pd.read_csv(full_csv_path)
print(f"✅ Dataset successfully loaded! Named 'df', Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

✅ Dataset successfully loaded! Named 'df', Shape: 11914 rows, 16 columns.


In [26]:
# Display the first 5 records to see our column properties and labels
df.head()

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


In [27]:
df.shape # (rows, columns)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 11914
Number of columns: 16


In [28]:
df.sample(15) # Random 15 rows

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
4948,Nissan,Frontier,2015,regular unleaded,261.0,6.0,MANUAL,four wheel drive,4.0,NaN,Compact,Crew Cab Pickup,21,16,2009,31510
2176,Chevrolet,Camaro,2015,premium unleaded (recommended),426.0,8.0,MANUAL,rear wheel drive,2.0,High-Performance,Midsize,Coupe,24,16,1385,37305
164,BMW,3 Series,2017,diesel,180.0,4.0,AUTOMATIC,all wheel drive,4.0,"Diesel,Luxury",Midsize,Wagon,40,30,3916,44450
9687,Mercedes-Benz,SLS AMG GT,2013,premium unleaded (required),583.0,8.0,AUTOMATED_MANUAL,rear wheel drive,2.0,"Exotic,Factory Tuner,Luxury,High-Performance",Compact,Coupe,19,13,617,199500
1054,Audi,A3,2017,premium unleaded (recommended),220.0,4.0,AUTOMATED_MANUAL,all wheel drive,2.0,Luxury,Compact,Convertible,34,25,3105,40300
9750,Hyundai,Sonata,2017,regular unleaded,245.0,4.0,AUTOMATIC,front wheel drive,4.0,Performance,Midsize,Sedan,31,22,1439,34350
5865,Subaru,Impreza,2015,regular unleaded,148.0,4.0,AUTOMATIC,all wheel drive,4.0,NaN,Compact,Sedan,37,28,640,20995
3701,Mercedes-Benz,E-Class,2016,premium unleaded (required),302.0,6.0,AUTOMATIC,all wheel drive,4.0,"Luxury,Performance",Midsize,Sedan,28,20,617,55600
1746,Mazda,B-Series,2001,regular unleaded,150.0,6.0,MANUAL,rear wheel drive,2.0,NaN,Compact,Regular Cab Pickup,21,15,586,15125
11646,Scion,xD,2013,regular unleaded,128.0,4.0,AUTOMATIC,front wheel drive,4.0,Hatchback,Compact,4dr Hatchback,33,27,105,16545


In [29]:
df.tail() #Displays the last 5 rows of the DataFrame df.

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
11909,Acura,ZDX,2012,premium unleaded (required),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,46120
11910,Acura,ZDX,2012,premium unleaded (required),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,56670
11911,Acura,ZDX,2012,premium unleaded (required),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,50620
11912,Acura,ZDX,2013,premium unleaded (recommended),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,50920
11913,Lincoln,Zephyr,2006,regular unleaded,221.0,6.0,AUTOMATIC,front wheel drive,4.0,Luxury,Midsize,Sedan,26,17,61,28995


In [30]:
df.dtypes #Displays the data type of each column in the DataFrame.

Make                     str
Model                    str
Year                   int64
Engine Fuel Type         str
Engine HP            float64
Engine Cylinders     float64
Transmission Type        str
Driven_Wheels            str
Number of Doors      float64
Market Category          str
Vehicle Size             str
Vehicle Style            str
highway MPG            int64
city mpg               int64
Popularity             int64
MSRP                   int64
dtype: object

In [31]:
df.columns #Returns a list (Index object) containing the names of all columns in the DataFrame.

Index(['Make', 'Model', 'Year', 'Engine Fuel Type', 'Engine HP',
       'Engine Cylinders', 'Transmission Type', 'Driven_Wheels',
       'Number of Doors', 'Market Category', 'Vehicle Size', 'Vehicle Style',
       'highway MPG', 'city mpg', 'Popularity', 'MSRP'],
      dtype='str')

### 🧹 Data Preprocessing & Hygiene

In [32]:
df.isna().sum() # Count missing values per column

Make                    0
Model                   0
Year                    0
Engine Fuel Type        3
Engine HP              69
Engine Cylinders       30
Transmission Type       0
Driven_Wheels           0
Number of Doors         6
Market Category      3742
Vehicle Size            0
Vehicle Style           0
highway MPG             0
city mpg                0
Popularity              0
MSRP                    0
dtype: int64

#### 🏷️ Handling `Market Category` Missing Values

`Market Category` is missing in ~31% of rows — too large a portion to 
safely impute with a guessed value (e.g., most-frequent), since that 
would fabricate a substantial chunk of the dataset. Instead, we label 
missing entries explicitly as `'Unknown'`, treating "no category listed" 
as its own honest, meaningful value rather than pretending to know 
something we don't.

All other missing values (`Engine Fuel Type`, `Engine HP`, 
`Engine Cylinders`, `Number of Doors`) are small (under 0.6% each) and 
will be handled automatically inside the preprocessing pipeline via 
`SimpleImputer`.

In [33]:
df['Market Category'] = df['Market Category'].fillna('Unknown')

In [34]:
df.describe(include='number').T #Generates a complete statistical summary of the DataFrame.

,count,mean,std,min,25%,50%,75%,max
Year,11914.0,2010.384338,7.579740,1990.0,2007.0,2015.0,2016.00,2017.0
Engine HP,11845.0,249.386070,109.191870,55.0,170.0,227.0,300.00,1001.0
Engine Cylinders,11884.0,5.628829,1.780559,0.0,4.0,6.0,6.00,16.0
Number of Doors,11908.0,3.436093,0.881315,2.0,2.0,4.0,4.00,4.0
highway MPG,11914.0,26.637485,8.863001,12.0,22.0,26.0,30.00,354.0
city mpg,11914.0,19.733255,8.987798,7.0,16.0,18.0,22.00,137.0
Popularity,11914.0,1554.911197,1441.855347,2.0,549.0,1385.0,2009.00,5657.0
MSRP,11914.0,40594.737032,60109.103604,2000.0,21000.0,29995.0,42231.25,2065902.0


In [35]:
df.info() #Displays a summary of the DataFrame structure.

<class 'pandas.DataFrame'>
RangeIndex: 11914 entries, 0 to 11913
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Make               11914 non-null  str    
 1   Model              11914 non-null  str    
 2   Year               11914 non-null  int64  
 3   Engine Fuel Type   11911 non-null  str    
 4   Engine HP          11845 non-null  float64
 5   Engine Cylinders   11884 non-null  float64
 6   Transmission Type  11914 non-null  str    
 7   Driven_Wheels      11914 non-null  str    
 8   Number of Doors    11908 non-null  float64
 9   Market Category    11914 non-null  str    
 10  Vehicle Size       11914 non-null  str    
 11  Vehicle Style      11914 non-null  str    
 12  highway MPG        11914 non-null  int64  
 13  city mpg           11914 non-null  int64  
 14  Popularity         11914 non-null  int64  
 15  MSRP               11914 non-null  int64  
dtypes: float64(3), int64(5), str(8)
m

In [36]:
df.dtypes # This command displays the data types of each column in the DataFrame df.

Make                     str
Model                    str
Year                   int64
Engine Fuel Type         str
Engine HP            float64
Engine Cylinders     float64
Transmission Type        str
Driven_Wheels            str
Number of Doors      float64
Market Category          str
Vehicle Size             str
Vehicle Style            str
highway MPG            int64
city mpg               int64
Popularity             int64
MSRP                   int64
dtype: object

In [38]:
print("Number of duplicate rows before dropping:", df.duplicated().sum())
df.drop_duplicates(inplace=True) # Remove duplicate rows from the DataFrame df.
print("Number of duplicate rows after dropping:", df.duplicated().sum()) # After dropping duplicates, check again to confirm that there are no duplicate rows remaining in the DataFrame df.

Number of duplicate rows before dropping: 715
Number of duplicate rows after dropping: 0


In [42]:
# ============================================================
# 🧹 COLUMN NAME SANITIZATION
# ============================================================
print("❓Original columns:" + str(df.columns.tolist()))
print()

df.columns = (
    df.columns
    .str.lower()                                        # lowercase everything
    .str.strip()                                        # remove leading/trailing whitespace
    .str.replace(r"[^a-z0-9]+", "_", regex=True)       # replace anything not a letter/number with _
    .str.strip("_")                                     # remove leading/trailing underscores
)

print("✅ Columns sanitized:" + str(df.columns.tolist()))

❓Original columns:['make', 'model', 'year', 'engine_fuel_type', 'engine_hp', 'engine_cylinders', 'transmission_type', 'driven_wheels', 'number_of_doors', 'market_category', 'vehicle_size', 'vehicle_style', 'highway_mpg', 'city_mpg', 'popularity', 'msrp']

✅ Columns sanitized:['make', 'model', 'year', 'engine_fuel_type', 'engine_hp', 'engine_cylinders', 'transmission_type', 'driven_wheels', 'number_of_doors', 'market_category', 'vehicle_size', 'vehicle_style', 'highway_mpg', 'city_mpg', 'popularity', 'msrp']


#### 🔍 Investigating Outliers and Suspicious Values

Before cleaning further, we investigate the extreme values found in 
`.describe()` — unusually high MPG, MSRP, and zero-cylinder entries — 
to understand whether they're genuine data points (e.g., electric 
vehicles) or errors.

In [43]:
# Check the extreme MPG entries
print(df[df['highway_mpg'] > 100][['make', 'model', 'year', 'engine_cylinders', 'highway_mpg', 'city_mpg']])
print()

# Check zero-cylinder entries
print(df[df['engine_cylinders'] == 0][['make', 'model', 'year', 'engine_fuel_type', 'engine_cylinders']].head(10))
print()

# Check the most expensive cars
print(df.nlargest(5, 'msrp')[['make', 'model', 'year', 'msrp']])

            make     model  year  engine_cylinders  highway_mpg  city_mpg
539         FIAT      500e  2015               0.0          108       122
540         FIAT      500e  2016               0.0          103       121
541         FIAT      500e  2017               0.0          103       121
1119        Audi        A6  2017               4.0          354        24
1983   Chevrolet   Bolt EV  2017               NaN          110       128
1984   Chevrolet   Bolt EV  2017               NaN          110       128
3716  Volkswagen    e-Golf  2015               NaN          105       126
3717  Volkswagen    e-Golf  2015               NaN          105       126
3718  Volkswagen    e-Golf  2016               NaN          105       126
3719  Volkswagen    e-Golf  2016               NaN          105       126
4705       Honda    Fit EV  2013               0.0          105       132
4706       Honda    Fit EV  2014               0.0          105       132
5780  Mitsubishi    i-MiEV  2017      